# Annulus Specialist Decoder

This notebook trains the current best Stage 1 model, then trains a Stage 2 specialist decoder only on annulus examples. Outputs are written to `outputs/annulus_specialist_decoder/`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from config import Stage1ModelConfig, Stage2ModelConfig, StageTrainingConfig, TwoStageRunConfig, TwoStageStackConfig
from datasets import build_two_stage_datasets, save_two_stage_dataset
from models import Stage1Regressor, Stage2CoordConvDecoder, evaluate_regression_predictions, evaluate_stage2_predictions, fit_stage1_model, fit_stage2_model, predict_stage1_coefficients, predict_stage2_logits, select_best_stage2_threshold, set_torch_seed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
set_torch_seed(SEED)

OUTPUT_ROOT = ROOT / 'outputs' / 'annulus_specialist_decoder'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

run_config = TwoStageRunConfig(
    N=8,
    training_samples=10000,
    validation_samples=2000,
    test_samples=500,
    rho=0.8,
    grid_size=32,
    threshold=0.5,
    use_validation_threshold_sweep=True,
    noise_level=0.01,
    seed=SEED,
    training_shape_weights=(('rectangle', 0.30), ('two_circles', 0.30), ('annulus', 0.15), ('ellipse', 0.15), ('circle', 0.10)),
    model=TwoStageStackConfig(
        stage1=Stage1ModelConfig(
            hidden_layer_sizes=(256, 512, 256),
            dropout_rates=(0.15, 0.15, 0.15),
            training=StageTrainingConfig(epochs=220, batch_size=64, learning_rate=0.0004, validation_frequency=10, verbose=False, early_stopping_patience=12, lr_drop_factor=0.5, lr_drop_period=60, weight_decay=0.00001, gradient_clip_norm=1.0),
        ),
        stage2=Stage2ModelConfig(
            hidden_layer_sizes=(512, 1024),
            dropout_rates=(0.10, 0.10),
            model_type='coord_conv_decoder',
            latent_grid_size=16,
            latent_channels=160,
            decoder_channels=(160, 128, 96, 64, 32),
            use_rectangle_edge_weighting=True,
            use_foreground_pos_weight=False,
            rectangle_edge_weight=4.0,
            rectangle_edge_width=3,
            edge_weight_mode='rectangle',
            annulus_edge_weight=1.0,
            annulus_edge_width=3,
            training=StageTrainingConfig(epochs=170, batch_size=96, learning_rate=0.0005, validation_frequency=60, verbose=False, early_stopping_patience=24, min_epochs=50, min_improvement=0.002, lr_drop_factor=0.5, lr_drop_period=80, weight_decay=0.00025, gradient_clip_norm=0.8, loss_type='bce_dice', dice_loss_weight=1.0, dice_smooth=1.0),
        ),
    ),
    output_dir=OUTPUT_ROOT,
)

run_output_dir = run_config.run_output_dir
run_output_dir.mkdir(parents=True, exist_ok=True)
dataset_bundle = build_two_stage_datasets(run_config)
dataset_paths = save_two_stage_dataset(dataset_bundle, run_output_dir / 'datasets')

stage1_model = Stage1Regressor(run_config.gradient_feature_size, run_config.coefficient_size, run_config.model.stage1.hidden_layer_sizes, run_config.model.stage1.dropout_rates)
stage1_training = run_config.model.stage1.training
stage1_result = fit_stage1_model(
    model=stage1_model,
    train_features=dataset_bundle.train.gradient_data,
    train_targets=dataset_bundle.train.coefficients,
    val_features=dataset_bundle.validation.gradient_data,
    val_targets=dataset_bundle.validation.coefficients,
    epochs=stage1_training.epochs,
    batch_size=stage1_training.batch_size,
    learning_rate=stage1_training.learning_rate,
    device=DEVICE,
    validation_frequency=stage1_training.validation_frequency,
    verbose=stage1_training.verbose,
    early_stopping_patience=stage1_training.early_stopping_patience,
    lr_drop_factor=stage1_training.lr_drop_factor,
    lr_drop_period=stage1_training.lr_drop_period,
    weight_decay=stage1_training.weight_decay,
    gradient_clip_norm=stage1_training.gradient_clip_norm,
)

pred_train = predict_stage1_coefficients(stage1_model, dataset_bundle.train.gradient_data, DEVICE, stage1_result)
pred_val = predict_stage1_coefficients(stage1_model, dataset_bundle.validation.gradient_data, DEVICE, stage1_result)
pred_test = predict_stage1_coefficients(stage1_model, dataset_bundle.test.gradient_data, DEVICE, stage1_result)
pred_fixed = predict_stage1_coefficients(stage1_model, dataset_bundle.fixed.gradient_data, DEVICE, stage1_result)

def select_shape(features, masks, shape_types, target_shape):
    indices = [i for i, shape_type in enumerate(shape_types) if shape_type == target_shape]
    return features[indices], masks[indices], tuple(shape_types[i] for i in indices)

train_features, train_masks, train_types = select_shape(pred_train, dataset_bundle.train.masks, dataset_bundle.train.shape_types, 'annulus')
val_features, val_masks, val_types = select_shape(pred_val, dataset_bundle.validation.masks, dataset_bundle.validation.shape_types, 'annulus')
test_features, test_masks, test_types = select_shape(pred_test, dataset_bundle.test.masks, dataset_bundle.test.shape_types, 'annulus')
fixed_features, fixed_masks, fixed_types = select_shape(pred_fixed, dataset_bundle.fixed.masks, dataset_bundle.fixed.shape_types, 'annulus')

specialist = Stage2CoordConvDecoder(run_config.coefficient_size, run_config.mask_pixels, run_config.model.stage2.hidden_layer_sizes, run_config.model.stage2.dropout_rates, run_config.model.stage2.latent_grid_size, run_config.model.stage2.latent_channels, run_config.model.stage2.decoder_channels)
stage2_training = run_config.model.stage2.training
stage2_result = fit_stage2_model(
    model=specialist,
    train_features=train_features,
    train_targets=train_masks,
    val_features=val_features,
    val_targets=val_masks,
    epochs=stage2_training.epochs,
    batch_size=stage2_training.batch_size,
    learning_rate=stage2_training.learning_rate,
    device=DEVICE,
    validation_frequency=stage2_training.validation_frequency,
    verbose=stage2_training.verbose,
    early_stopping_patience=stage2_training.early_stopping_patience,
    train_shape_types=train_types,
    grid_size=run_config.grid_size,
    use_rectangle_edge_weighting=False,
    min_epochs=stage2_training.min_epochs,
    min_improvement=stage2_training.min_improvement,
    lr_drop_factor=stage2_training.lr_drop_factor,
    lr_drop_period=stage2_training.lr_drop_period,
    weight_decay=stage2_training.weight_decay,
    gradient_clip_norm=stage2_training.gradient_clip_norm,
    loss_type=stage2_training.loss_type,
    dice_loss_weight=stage2_training.dice_loss_weight,
    dice_smooth=stage2_training.dice_smooth,
    use_foreground_pos_weight=False,
)

pred_val_masks = predict_stage2_logits(specialist, val_features, DEVICE, stage2_result)
pred_test_masks = predict_stage2_logits(specialist, test_features, DEVICE, stage2_result)
pred_fixed_masks = predict_stage2_logits(specialist, fixed_features, DEVICE, stage2_result)
threshold_summary = select_best_stage2_threshold(val_masks, pred_val_masks, run_config.threshold_candidates)
threshold_summary['selection_mode'] = 'validation_sweep'
threshold = float(threshold_summary['selected_threshold'])
summary = {
    'shape': 'annulus',
    'stage1_metrics': {
        'test': evaluate_regression_predictions(dataset_bundle.test.coefficients, pred_test),
        'fixed': evaluate_regression_predictions(dataset_bundle.fixed.coefficients, pred_fixed),
    },
    'specialist_metrics': {
        'test': evaluate_stage2_predictions(test_masks, pred_test_masks, threshold),
        'fixed': evaluate_stage2_predictions(fixed_masks, pred_fixed_masks, threshold),
    },
    'threshold_summary': threshold_summary,
    'dataset_paths': {key: str(value) for key, value in dataset_paths.items()},
    'stage1_history': stage1_result.history,
    'stage2_history': stage2_result.history,
}
with (run_output_dir / 'summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)
print('Annulus specialist test IoU:', summary['specialist_metrics']['test']['mean_iou'])
print('Annulus specialist fixed IoU:', summary['specialist_metrics']['fixed']['mean_iou'])
